In [1]:
import sys

In [2]:
def read_input():
    data = list(map(int, sys.stdin.read().split()))
    idx = 0
    n = data[idx]; idx += 1
    c_min = 10**7
    dist = [[0] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            dist[i][j] = data[idx]
            idx += 1
            if i != j:
                c_min = min(c_min, dist[i][j])
    return n, dist, c_min

In [3]:
### 1. SOLUTION USING BACKTRACKING
def backtrack(n, dist, c_min):
    f = 0
    f_best = 10**7
    visited = [False] * n
    visited[0] = True
    x = [0] * n
    x_best = [0] * n
    
    def Try(k):
        nonlocal f, f_best, visited, x, x_best
        for v in range(1, n):
            if visited[v]:
                continue
            visited[v] = True
            x[k] = v
            f += dist[x[k-1]][x[k]]
            if k == n - 1:
                if f + dist[x[k]][x[0]] < f_best:
                    f_best = f + dist[x[k]][x[0]]
                    x_best = x.copy()
            else:
                if (n - k + 1) * c_min + f < f_best:
                    Try(k + 1)
            # backtrack
            visited[v] = False
            f -= dist[x[k-1]][x[k]]
    Try(1)
    return x_best, f_best

In [4]:
import sys
import random
import time

def read_input():
    data = list(map(int, sys.stdin.read().split()))
    if not data:
        return 0, [], 0
    idx = 0
    n = data[idx]; idx += 1
    c_min = 10**7
    dist = [[0] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            dist[i][j] = data[idx]
            idx += 1
            if i != j:
                c_min = min(c_min, dist[i][j])
    return n, dist, c_min

### 2. SOLUTION USING LOCAL SEARCH
def calculate_obj(n, dist, sol):
    res = 0
    for i in range(1, n):
        res += dist[sol[i-1]][sol[i]]
    res += dist[sol[n-1]][sol[0]]
    return res

def local_search(n, dist):
    # Khởi tạo và CỐ ĐỊNH đỉnh 0 ở vị trí đầu tiên
    current_sol = list(range(n))
    random.shuffle(current_sol)
    idx_zero = current_sol.index(0)
    current_sol[0], current_sol[idx_zero] = current_sol[idx_zero], current_sol[0]
    
    best_obj = calculate_obj(n, dist, current_sol)

    improved = True
    while improved:
        improved = False

        ### 2.1. SWAP
        for a in range(1, n - 1):
            for b in range(a + 1, n): # Tối ưu: b chạy từ a+1
                # Hoán đổi trực tiếp (In-place)
                current_sol[a], current_sol[b] = current_sol[b], current_sol[a]
                cand_obj = calculate_obj(n, dist, current_sol)

                if cand_obj < best_obj:
                    best_obj = cand_obj
                    improved = True
                    break
                else:
                    # Hoàn tác nếu không tốt hơn
                    current_sol[a], current_sol[b] = current_sol[b], current_sol[a]
            if improved:
                break
        if improved:
            continue

        ### 2.2. INSERT
        for a in range(1, n):
            for b in range(1, n):
                if a == b: 
                    continue
                
                # Rút a chèn vào b
                val = current_sol.pop(a)
                current_sol.insert(b, val)
                cand_obj = calculate_obj(n, dist, current_sol)
                
                if cand_obj < best_obj:
                    best_obj = cand_obj
                    improved = True
                    break
                else:
                    # Hoàn tác
                    current_sol.pop(b)
                    current_sol.insert(a, val)
            if improved:
                break
        if improved:
            continue

        ### 2.3. 2OPT
        for a in range(1, n - 1):
            for b in range(a + 1, n): # Tối ưu: b chạy từ a+1
                # Đảo ngược đoạn a->b
                current_sol[a:b + 1] = reversed(current_sol[a:b + 1])
                cand_obj = calculate_obj(n, dist, current_sol)

                if cand_obj < best_obj:
                    best_obj = cand_obj
                    improved = True
                    break
                else:
                    # Hoàn tác (Đảo ngược thêm lần nữa)
                    current_sol[a:b + 1] = reversed(current_sol[a:b + 1])
            if improved:
                break

    return current_sol, best_obj

def multi_start_local_search(n, dist, num_ini):
    start_time = time.time()
    best_obj = float('inf')
    best_sol = None
    
    for _ in range(num_ini):
        # Sửa logic kiểm tra thời gian
        if time.time() - start_time > 15.0:
            break
            
        current_sol, current_obj = local_search(n, dist)
        
        if current_obj < best_obj:
            best_obj = current_obj
            # Phải copy lại nghiệm tốt nhất vì current_sol ở mảng local_search sẽ bị hủy
            best_sol = current_sol.copy() 
            
    return best_sol, best_obj

In [14]:
### 3. USING PYWRAPCP
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2 

def solve_pywrapcp(n, dist):
    manager = pywrapcp.RoutingIndexManager(len(dist), 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return dist[from_node][to_node]

    transit_distance_callback = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_distance_callback)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.FromSeconds(5)
    solution = routing.SolveWithParameters(search_parameters)

    status = routing.status()
    if status != 1:
        print(-1)
        return

    if solution:
        print(solution.ObjectiveValue())
        index = routing.Start(0)
        route = []
        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))
        route.append(manager.IndexToNode(index))
        print(*route)

In [17]:
def main():
    f = open("input.txt", "r")
    sys.stdin = f
    n, dist, c_min = read_input()
    # f_best, x_best = backtrack(n, dist, c_min)
    # x_best, f_best = multi_start_local_search(n, dist, 20)
    solve_pywrapcp(n, dist)
    # print(n)
    # print(f_best)
    # for v in x_best:
    #     print(f"{v + 1}", end=" ")
    f.close()

In [18]:
main()

72713
0 29 23 41 1 28 37 27 21 25 13 4 40 44 45 14 20 42 15 49 12 33 2 19 47 10 22 7 38 46 5 8 48 30 36 43 31 34 3 24 18 11 32 39 16 26 6 9 35 17 0


In [ ]:
lst = [1, 2, 3, 4]
lst.insert(3, 5)
print(lst)